# 01 · Metabolic dropout diagnostic (Kolla et al. 2020, E16)

Before building `metabolic_metacells`, this notebook answers three questions:

1. **Can raw UMI counts be recovered** from the log-normalised matrix? (Needed for any Poisson/depth-based method.)
2. **Which metabolic genes matter for reaction scores?** Each gene gets a *dropout leverage*: how much ECS features change if that gene alone reads zero, using the same AND/OR rules as `calculate_ecs`.
3. **Is the dropout in high-leverage genes caused by shallow sequencing (fixable by pooling) or by genuine on/off expression (pooling would blur it)?** And how many cells would need pooling to detect each gene?

Outputs are written as CSVs to `OUT_DIR`.

## 0 · Settings
Edit the paths below. On the HPC, keep large data files in your project or scratch space rather than your home directory.

In [ ]:
import os, sys, json

REPO_DIR = os.path.expanduser('~/Metabolic-pipeline')      # where you cloned the repository
DATA_PATH = '/path/to/8b0e6d9a.h5ad'                        # Kolla E16 file
OUT_DIR = os.path.expanduser('~/metabolic_results/01_diagnostic_E16')

CELLTYPE_COL = 'cell_type'
SYMBOL_COL = 'gene_symbol'      # adata.var column with gene symbols; var_names are used if it doesn't exist
SPECIES = 'mmusculus'
AND_STRATEGY = 'median'
OR_STRATEGY = 'sum'
REFERENCE_SIZE = 50             # current SEACells target_metacell_size

sys.path.insert(0, REPO_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from metabolic_tools.metacell_diagnostics import recover_counts, gene_dropout_leverage, dropout_diagnostic

adata = sc.read_h5ad(DATA_PATH)
print(adata)
print('\nCell types:\n', adata.obs[CELLTYPE_COL].value_counts())
adata.var.head()

## 1 · Recover raw counts
If `adata.X = log1p(counts / library_size × constant)`, then within each cell the smallest non-zero value of `expm1(X)` corresponds to 1 UMI. Dividing by it should give whole numbers — this is **checked**, not assumed.

What to look for:
- `integer_fraction` ≈ 1.0 → recovery is exact.
- `normalised_total_cv` ≈ 0 → every cell was scaled to the same total.
- `pearson_r_vs_...` ≈ 1.0 (if the file carries a library size column) → recovered depth matches the original.

If this raises an error, the matrix was transformed in a way that can't be reversed and we'll need the raw matrices from GEO.

In [ ]:
report = recover_counts(adata)
print(json.dumps(report, indent=2))

fig, ax = plt.subplots(figsize=(6, 4))
for ct, lib in adata.obs.groupby(CELLTYPE_COL, observed=True)['total_counts_recovered']:
    ax.hist(np.log10(lib), bins=50, histtype='step', label=ct)
ax.set_xlabel('log10 recovered UMIs per cell')
ax.set_ylabel('cells')
ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()

## 2 · Gene dropout leverage
Evaluated on each cell type's mean expression, with the AND/OR rules `calculate_ecs` uses. Rough guide:
- single-gene rule (or a split isozyme branch) → 1 per feature
- isozyme under OR=sum without splitting → that isozyme's share of the total
- 2-subunit complex under AND=median → about ½
- large complex under AND=median → close to 0

The `min` run is included only for comparison, to show how much the choice of AND operator changes which genes matter.

In [ ]:
leverage, adata_model = gene_dropout_leverage(
    adata, CELLTYPE_COL, species=SPECIES, symbol_col=SYMBOL_COL,
    and_strategy=AND_STRATEGY, or_strategy=OR_STRATEGY)
leverage_min, _ = gene_dropout_leverage(
    adata, CELLTYPE_COL, species=SPECIES, symbol_col=SYMBOL_COL,
    and_strategy='min', or_strategy=OR_STRATEGY)

cell_types = [c for c in leverage.columns if c in set(adata.obs[CELLTYPE_COL].astype(str))]
print(f'{len(leverage)} model genes scored across {len(cell_types)} cell types')

comparison = pd.DataFrame({
    'median: total leverage': leverage[cell_types].mean(axis=1),
    'min: total leverage': leverage_min[cell_types].mean(axis=1),
}).join(leverage[['symbol'] + [c for c in leverage.columns if c.startswith('n_')]])
comparison.sort_values('median: total leverage', ascending=False).head(25)

In [ ]:
# How leverage is spread across GPR categories: which rule types dominate dropout sensitivity?
cat_cols = [c for c in leverage.columns if c.startswith('n_') and c != 'n_features']
dominant = leverage[cat_cols].idxmax(axis=1).str.removeprefix('n_')
pd.DataFrame({
    'genes': dominant.value_counts(),
    'leverage share (median)': leverage[cell_types].mean(axis=1).groupby(dominant).sum() / leverage[cell_types].mean(axis=1).sum(),
    'leverage share (min)': leverage_min[cell_types].mean(axis=1).groupby(dominant).sum() / leverage_min[cell_types].mean(axis=1).sum(),
}).round(3)

## 3 · Dropout: sequencing depth or biology?
For each gene and cell type:
- **expected_zero_frac**: zeros expected from depth alone (Poisson).
- **excess_zero_frac**: observed minus expected. Near 0 → pooling will fix it. Large → genuinely on in some cells, off in others.
- **units_needed**: how many median-depth cells must be pooled to detect the gene with 95% probability.

The summary weights every gene by its leverage, so it reflects the genes that actually move reaction scores.

In [ ]:
gene_df, summary = dropout_diagnostic(adata_model, leverage, CELLTYPE_COL, reference_size=REFERENCE_SIZE)
summary.round(3)

In [ ]:
groups = list(summary.index)
ncol = min(4, len(groups))
nrow = int(np.ceil(len(groups) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.2 * nrow), squeeze=False)
for ax, group in zip(axes.flat, groups):
    d = gene_df[(gene_df['group'] == group) & (gene_df['leverage'] > 0)]
    sca = ax.scatter(d['expected_zero_frac'], d['zero_frac'], s=4 + 20 * d['leverage'] / d['leverage'].max(),
                     c=d['leverage'], cmap='viridis', alpha=0.6, linewidths=0)
    ax.plot([0, 1], [0, 1], color='grey', lw=0.8, ls='--')
    ax.set_title(group, fontsize=9)
    ax.set_xlabel('expected zeros (depth only)', fontsize=8)
    ax.set_ylabel('observed zeros', fontsize=8)
for ax in axes.flat[len(groups):]:
    ax.axis('off')
fig.colorbar(sca, ax=axes, shrink=0.6, label='leverage')
plt.show()

In [ ]:
# Leverage-weighted distribution of cells needed to detect each gene
fig, ax = plt.subplots(figsize=(6, 4))
bins = np.logspace(0, 4, 50)
for group in groups:
    d = gene_df[(gene_df['group'] == group) & np.isfinite(gene_df['units_needed'])]
    ax.hist(d['units_needed'].clip(upper=bins[-1]), bins=bins, weights=d['leverage'], histtype='step', label=group)
ax.axvline(REFERENCE_SIZE, color='grey', ls='--', lw=0.8)
ax.set_xscale('log')
ax.set_xlabel('median-depth cells needed for 95% detection')
ax.set_ylabel('leverage-weighted genes')
ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()

## 4 · Save

In [ ]:
with open(os.path.join(OUT_DIR, 'count_recovery_report.json'), 'w') as f:
    json.dump(report, f, indent=2)
leverage.to_csv(os.path.join(OUT_DIR, 'gene_leverage_median.csv'))
leverage_min.to_csv(os.path.join(OUT_DIR, 'gene_leverage_min.csv'))
gene_df.to_csv(os.path.join(OUT_DIR, 'gene_dropout.csv'), index=False)
summary.to_csv(os.path.join(OUT_DIR, 'dropout_summary.csv'))
print('Saved to', OUT_DIR)